In [1]:
import torch
from torch import nn
from torch.distributions import Categorical
from environment import Environment
import numpy as np
import torch.nn.functional as F
import torch.optim as optim
from itertools import count

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
env = Environment(custom_datapath='./data/00000.csv')

2024-05-24 11:46:56.749345741 [E:onnxruntime:Default, provider_bridge_ort.cc:1480 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1193 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.11: cannot open shared object file: No such file or directory

2024-05-24 11:46:56.749374751 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:747 CreateExecutionProviderInstance] Failed to create CUDAExecutionProvider. Please reference https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements to ensure all dependencies are met.


In [4]:
num_actions = 50
action_range = np.linspace(-2, 2, num_actions)

In [5]:
class Critic(nn.Module):
    def __init__(self, in_features):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(in_features, 128) 
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 1) 
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
def train_only_critic():
    R = 0
    returns = []
    
    for r in rewards[::-1]:
        R = r + 0.99 * R # Gamma is 0.99
        returns.insert(0, R)

    returns = torch.tensor(returns, device=device)
    returns = (returns - returns.mean()) / (returns.std() + eps)

    old_probs, state_values, states, actions = zip(*saved_actions)
    state_values = torch.stack(state_values).to(device)

    critic_loss = F.smooth_l1_loss(state_values.squeeze(), returns)
    critic_optimizer.zero_grad()
    critic_loss.backward(retain_graph=False)
    critic_optimizer.step()

    print(critic_loss.item())

    del rewards[:]
    del saved_actions[:]

In [9]:
def finish_episode():
    # Calculating losses and performing backprop
    R = 0
    returns = []
    epsilon = 0.2
    num_epochs = 1

    for r in rewards[::-1]:
        R = r + 0.9 * R # Gamma is 0.99
        returns.insert(0, R)

    returns = torch.tensor(returns, device=device)
    returns = (returns - returns.mean()) / (returns.std() + eps)

    old_probs, state_values, states, actions = zip(*saved_actions)

    old_probs = torch.stack(old_probs).to(device)
    state_values = torch.stack(state_values).to(device)
    states = torch.stack(states).to(device)
    actions = torch.stack(actions).to(device)

    advantages = returns - state_values.squeeze()

    for epoch in range(num_epochs):

        new_probs = actor(states).gather(1, actions).squeeze()

        ratios = new_probs / old_probs

        surr1 = ratios * advantages
        surr2 = torch.clamp(ratios, 1 - epsilon, 1 + epsilon) * advantages

        #actor_loss = -torch.min(surr1, surr2).mean()
        actor_loss = -surr1.mean()

        actor_optimizer.zero_grad()
        actor_loss.backward(retain_graph=True)
        actor_optimizer.step()

        if epoch == num_epochs - 1:
            critic_loss = F.smooth_l1_loss(state_values.squeeze(), returns)
            critic_optimizer.zero_grad()
            critic_loss.backward(retain_graph=False)
            critic_optimizer.step()

    del rewards[:]
    del saved_actions[:]

In [10]:
def select_action(state_and_info):
    logits = actor(state_and_info.unsqueeze(0))
    probs = F.softmax(logits, dim=-1)
    distribution = Categorical(probs)
    action = distribution.sample()

    state_value = critic(state_and_info.unsqueeze(0))

    saved_actions.append((probs[0][action.item()].detach(), state_value, state_and_info, action.detach()))

    action_item = action.item()

    return action_item

In [11]:
def train(train_critic_only=False, n_episodes = 100):
    best_cost = 10000

    for i_episode in range(n_episodes):
        state, info = env.reset()

        integral = 0
        prev_error = 0
        prev_action = 0
        prev_derivative = 0

        while True:
            error = state[0] - state[1]
            integral += error
            derivative = error - prev_error

            action_idx = select_action(torch.tensor([*state, error, prev_error, integral, derivative, prev_derivative, prev_action], dtype=torch.float32, device=device))
            action = action_range[action_idx]

            next_state, cost, done, terminated, info = env.step(action)

            rewards.append(cost)

            state = next_state

            prev_error = error
            prev_derivative = derivative
            prev_action = action

            if done:
                if train_critic_only:
                    train_only_critic()
                else:
                    total_cost = env.get_total_cost()
                    if total_cost[2] < best_cost: best_cost = total_cost[2]
                    print(f'Episode {i_episode} Total Cost:', total_cost)
                    finish_episode()
                break

    return best_cost

In [14]:
actor = torch.load('./models/mlp_pid.pth', map_location='cpu') # model takes: 'target_lataccel', 'current_lataccel', 'vEgo', 'aEgo', 'roll', 'error', 'prev_error', 'integral', 'derivative', 'prev_derivative', 'prev_action'
actor.to(device)
actor.eval()
actor_optimizer = optim.Adam(actor.parameters(), lr=4e-6)
saved_actions = []
rewards = []

critic = Critic(in_features=11).to(device)
critic_optimizer = optim.Adam(critic.parameters(), lr=3e-2)
eps = np.finfo(np.float32).eps.item()

In [16]:
train(train_critic_only=True, n_episodes=20)
best_cost = train(n_episodes=500)

0.34048499715793046
0.47318298260722663
0.3488665204359057
0.35956365627741704


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fcfdc0e94c0>>
Traceback (most recent call last):
  File "/home/adesbiens31/.pyenv/versions/3.9.19/envs/deeplearning/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


0.3048526962538443
0.353430970852853
0.3276649719055961
0.3380600064821419
0.328624113592767
0.3556571571852313
0.3392398471456146
0.3780994848912307
0.34254830452268437
0.3204211925758827
